# Gemini PDF Analysis Setup

This notebook demonstrates how to use the Gemini SDK to analyze PDF documents. We'll start by setting up the environment and initializing the Gemini client.

In [4]:
# Load env variables and create client
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
import time

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "requirements.txt").exists()),
    Path.cwd(),
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.gemini_retry import generate_content_with_retry

load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL_ID = "gemini-2.5-flash"

## Helper Functions

We'll use a standard `chat` function that includes retry logic for handling rate limits and other transient errors.

In [5]:
def chat(
    prompt,
    system_instruction=None,
    temperature=1.0,
    stop_sequences=None,
    tools=None,
    max_retries=5,
):
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        stop_sequences=stop_sequences,
        tools=tools,
    )
    
    response = generate_content_with_retry(
        client=client,
        model=MODEL_ID,
        contents=prompt,
        config=config,
        max_attempts=max_retries,
    )
    return response

## PDF Upload and Analysis

To analyze a PDF, we first upload it using the File API. Once uploaded, we can pass the file reference along with a text prompt to Gemini.

In [6]:
# 1. Upload the PDF file
pdf_path = "../../assets/pdfs/earth.pdf"
print(f"Uploading {pdf_path}...")
uploaded_file = client.files.upload(file=pdf_path)

# 2. Wait for the file to be processed (optional but recommended for large files)
while uploaded_file.state == "PROCESSING":
    print(".", end="", flush=True)
    time.sleep(2)
    uploaded_file = client.files.get(name=uploaded_file.name)

print(f"\nFile uploaded and processed: {uploaded_file.uri}")

# 3. Ask Gemini to analyze the PDF
prompt = "Summarize the main points of this document in 3-5 bullet points."
response = chat([uploaded_file, prompt])

print("\n--- Gemini Analysis ---")
print(response.text)

Uploading ../../assets/pdfs/earth.pdf...

File uploaded and processed: https://generativelanguage.googleapis.com/v1beta/files/rj7pop5sl3n2

--- Gemini Analysis ---
Here are the main points of the document in 4 bullet points:

*   Earth is the third planet from the Sun and the only known astronomical object to harbor life, primarily due to its extensive liquid surface water and dynamic atmosphere.
*   It formed approximately 4.5 billion years ago, has a liquid outer core that generates a protective magnetosphere, and its crust is composed of slowly moving tectonic plates.
*   The atmosphere, mainly nitrogen and oxygen, along with greenhouse gases, maintains a stable temperature suitable for liquid water, drives a global climate system, and protects the surface.
*   Earth is orbited by one permanent natural satellite, the Moon, and human activities are increasingly impacting its environment and biosphere.
